# AdaptAuth-ML v12: incorporación de variables de comportamiento por usuario y política de decisión escalonada

**Tesis:** Modelo de autenticación adaptativa basado en Machine Learning para la detección de accesos anómalos en PYMES del sector de servicios empresariales en Lima, 2026
**Tesista:** Ashly Herminia Jasmín Chávez Marón · **Asesor:** Mg. Daniel Alfonso Quinto Pazce
**Universidad Nacional Mayor de San Marcos**, Facultad de Ingeniería de Sistemas e Informática, Tesis II

---

## Resumen

El presente cuaderno documenta la evaluación de la versión 12 de AdaptAuth-ML. Respecto de la versión 11, se incorporan ocho variables que comparan cada inicio de sesión con el historial previo del mismo usuario y se reemplaza el bloqueo directo por una política escalonada que prioriza la verificación OTP. La variante de variables se seleccionó exclusivamente con el mes de validación y se evaluó una sola vez en el periodo de prueba de SuspiciousLogin (mayo y junio; 20 852 eventos). La versión 12 obtuvo un F1 de 0.667, con intervalo de confianza al 95 % de 0.620 a 0.712 por bootstrap de usuarios, lo que supera la meta de 0.60 con significancia estadística (p = 0.004). La política escalonada mantuvo la proporción de accesos sospechosos detenidos (53.7 %) y redujo a cero los bloqueos directos.

## 1. Introducción

La versión 11 de AdaptAuth-ML alcanzó un F1 puntual de 0.628 en SuspiciousLogin; sin embargo, el límite inferior de su intervalo de confianza (0.581) no superó la meta de 0.60, por lo que la hipótesis específica HE2 no pudo sostenerse con significancia. Las variables de la versión 11 describen el contexto del acceso (hora, red, ubicación, tipo de inicio), pero no lo comparan con el comportamiento habitual de cada usuario, aun cuando la literatura de autenticación basada en riesgo estima la probabilidad de cada atributo condicionada al historial del usuario (Freeman et al., 2016).

Asimismo, la política de tres niveles de la versión 11 bloqueaba directamente los accesos con riesgo superior a θ2. En un entorno empresarial, un bloqueo interrumpe la operación del usuario legítimo, mientras que una verificación OTP permite confirmar la identidad sin detener la operación.

**Objetivos de este experimento:**

1. Determinar si las variables de comportamiento por usuario elevan el F1 por encima de 0.60 con significancia estadística.
2. Determinar si una política escalonada que prioriza el OTP reduce los bloqueos sin disminuir la detección.

## 2. Materiales y métodos

### 2.1 Datos

Se utilizó SuspiciousLogin v1 (commit `68cf94b`), con una partición temporal fijada en versiones anteriores: entrenamiento de diciembre a marzo, validación en abril y prueba en mayo y junio. Se excluyen las variables `login_challenge_method`, `risk_score` y `risk_level`, por ser consecuencia de la evaluación de riesgo de la plataforma de origen.

In [1]:
import sys, os, json, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    subprocess.run("git clone -q https://github.com/AshlyChavez/adaptauth-ml.git && cd adaptauth-ml && git checkout -q e67f5a7", shell=True)
    subprocess.run("pip -q install scikit-learn==1.8.0", shell=True)
    subprocess.run("git clone -q https://github.com/salomaopena/SuspiciousLoginDataset.git sl && cd sl && git checkout -q 68cf94b", shell=True)
    REPO, DATA = Path("adaptauth-ml"), next(Path("sl").rglob("suspicious_logins_public_v1.csv"))
else:
    REPO, DATA = Path(os.environ.get("REPO", "..")).resolve(), Path(os.environ["DATA"])
sys.path.insert(0, str(REPO / "src")); sys.path.insert(0, str(REPO))

import numpy as np, pandas as pd, sklearn
from sklearn.metrics import average_precision_score, roc_auc_score
import adaptauth as aa
from modelo_adaptauth import ModeloAdaptAuth
from run_suspiciouslogin import preparar, VARIABLES_USUARIO
pd.set_option("display.precision", 3)

raw = pd.read_csv(DATA)
d, X12, _ = preparar(raw)                    # variables de la v12
_, X11, _ = preparar(raw, usuario=False)     # variables de la v11
mes = d["month"].replace({12: 0}).to_numpy()
tr, va, te = mes <= 3, mes == 4, np.isin(mes, [5, 6])
y = d["anomalo"].to_numpy()
pd.DataFrame({"Conjunto": ["Entrenamiento (dic-mar)", "Validación (abr)", "Prueba (may-jun)"],
              "Eventos": [tr.sum(), va.sum(), te.sum()], "Sospechosos": [y[tr].sum(), y[va].sum(), y[te].sum()],
              "% sospechosos": [100 * y[m].mean() for m in (tr, va, te)]}).round(2)

,Conjunto,Eventos,Sospechosos,% sospechosos
0,Entrenamiento (dic-mar),35969,2935,8.16
1,Validación (abr),12094,1002,8.29
2,Prueba (may-jun),20852,1577,7.56


### 2.2 Variables de comportamiento por usuario

Cada variable se calcula únicamente con los accesos **anteriores** del mismo usuario, por lo que no introduce fuga temporal. En operación, estos valores se obtienen de una tabla de historial por usuario que se actualiza en cada acceso.

**Tabla 1.** Variables incorporadas en la versión 12.

| Variable | Definición |
|---|---|
| `u_accesos_previos` | log(1 + n), donde n es el número de accesos previos del usuario |
| `u_frac_mismo_pais` | Fracción de accesos previos realizados desde el mismo país |
| `u_frac_mismo_tipo` | Fracción de accesos previos con el mismo tipo de inicio de sesión |
| `u_pais_nuevo` | 1 si el país no figura en el historial del usuario |
| `u_tipo_nuevo` | 1 si el tipo de inicio no figura en el historial del usuario |
| `u_desv_hora` | 1 − cos(hora actual − hora media circular previa) |
| `u_ratio_gap` | Horas desde el último acceso entre el intervalo medio previo, acotado a [0, 50] |
| `u_noche_inusual` | Acceso nocturno × (1 − fracción previa de accesos nocturnos) |

In [2]:
d.loc[te].assign(**X12.loc[te, VARIABLES_USUARIO]).groupby("anomalo")[VARIABLES_USUARIO].mean().T \
  .rename(columns={0: "Legítimos (media)", 1: "Sospechosos (media)"})

anomalo,Legítimos (media),Sospechosos (media)
u_accesos_previos,1.731,1.101
u_frac_mismo_pais,0.416,0.293
u_frac_mismo_tipo,0.321,0.258
u_tipo_nuevo,0.010,0.014
u_pais_nuevo,0.002,0.005
u_desv_hora,0.304,0.118
u_ratio_gap,0.780,0.741
u_noche_inusual,0.095,0.397


### 2.3 Protocolo de selección

Para evitar el sobreajuste al periodo de prueba, se fijó el siguiente protocolo antes de evaluar:

1. Se comparan tres variantes de variables **solo con el mes de validación**, mediante AUC-PR: (A) variables de la versión 11; (B) A más las variables por usuario; (C) B más dos variables de rareza del contexto (país × tipo de inicio y país × franja horaria, con frecuencias contadas solo en entrenamiento).
2. La variante elegida se evalúa **una sola vez** en el periodo de prueba.

### 2.4 Modelo, calibración y política de decisión

El clasificador no cambia respecto de la versión 11: HistGradientBoosting (semilla 42; 400 iteraciones; tasa de aprendizaje 0.05; 31 hojas; regularización L2 de 1.0). El umbral θ1 maximiza el F1 con FPR ≤ 5 % y θ2 cumple un FPR de bloqueo ≤ 0.1 %; ambos se calibran en validación.

La política escalonada de la versión 12 se define como:

- **Acceso directo** si R ≤ θ1.
- **Verificación OTP** si R > θ1, incluido el riesgo alto.
- **Bloqueo** únicamente si el OTP falla (dato disponible solo en operación) o si R > θ2 y existe viaje imposible.

Dado que el OTP y el bloqueo detienen el acceso por igual, las métricas de detección se calculan sobre la predicción R > θ1.

### 2.5 Análisis estadístico

- **Intervalo de confianza del F1:** bootstrap por conglomerados de usuario (2 000 réplicas), dado que los accesos de un mismo usuario no son independientes.
- **Contraste de la meta:** p = proporción de réplicas con F1 ≤ 0.60 (prueba unilateral).
- **Estabilidad:** cinco semillas del clasificador.
- **Comparación con la línea base de reglas calibrada al mismo FPR de validación:** bootstrap por usuario y McNemar exacto (ejecutados en `run_validacion_pi13.py` y `run_analisis_pi14.py`).

## 3. Resultados

### 3.1 Selección de la variante en validación

In [3]:
franja = pd.cut(d["event_hour"], [-1, 5, 11, 17, 23], labels=False)
def rareza(a, b):
    clave = d[a].astype(str) + "|" + b.astype(str)
    frec = clave[tr].value_counts(normalize=True)
    return -np.log(clave.map(frec).fillna(frec.min() / 10))
contexto = pd.DataFrame(dict(c_rareza_pais_tipo=rareza("ip_country", d["login_type"]),
                             c_rareza_pais_franja=rareza("ip_country", franja)), index=d.index)
variantes = {"A. Variables de la v11": X11, "B. A + variables por usuario": X12,
             "C. B + rareza del contexto": pd.concat([X12, contexto], axis=1)}
modelos, filas = {}, []
for nombre, X in variantes.items():
    m = ModeloAdaptAuth(seed=42).fit(X[tr].to_numpy(), y[tr]); modelos[nombre] = m
    filas.append(dict(Variante=nombre, Variables=X.shape[1],
                      **{"AUC-PR (validación)": average_precision_score(y[va], m.predict_proba(X[va].to_numpy())[:, 1])}))
seleccion = pd.DataFrame(filas); seleccion.round(4)

,Variante,Variables,AUC-PR (validación)
0,A. Variables de la v11,67,0.589
1,B. A + variables por usuario,75,0.682
2,C. B + rareza del contexto,77,0.682


**Tabla 2.** La variante B obtiene el mayor AUC-PR en validación. La variante C no la supera, por lo que se adopta B, la más simple de las dos. B constituye la versión 12.

### 3.2 Desempeño en el periodo de prueba

In [4]:
cfg = aa.Config(cold_start_otp=False, label="anomalo")
dv = d[va].reset_index(drop=True)
usuarios, inv = np.unique(d.loc[te, "actor_pseudo_id"].to_numpy(), return_inverse=True)

def bootstrap_f1(yy, pred, B=2000, seed=42):
    sm = lambda v: np.bincount(inv, weights=v.astype(float), minlength=len(usuarios))
    tp, fp, pos = sm((pred == 1) & (yy == 1)), sm((pred == 1) & (yy == 0)), sm(yy == 1)
    rng, f1 = np.random.default_rng(seed), []
    for _ in range(B):
        w = np.bincount(rng.integers(0, len(usuarios), len(usuarios)), minlength=len(usuarios)).astype(float)
        TP, FP, P = w @ tp, w @ fp, w @ pos
        f1.append(2 * TP / max(2 * TP + FP + (P - TP), 1e-9))
    return np.array(f1)

def evaluar(X, m, seed=42):
    cal = aa.calibrate(dv.assign(c_i=0.0), m.predict_proba(X[va].to_numpy())[:, 1], cfg, (1.0,))
    R = m.predict_proba(X[te].to_numpy())[:, 1]
    dec = aa.decide_escalonado(R, cal["theta1"], cal["theta2"], d.loc[te, "impossible_travel"].to_numpy())
    pred = (dec >= 1).astype(int); met = aa.binary_metrics(y[te], pred); f1b = bootstrap_f1(y[te], pred, seed=seed)
    return dict(R=R, dec=dec, cal=cal, met=met, auc=roc_auc_score(y[te], R), aucpr=average_precision_score(y[te], R),
                ic=np.quantile(f1b, [.025, .975]), p=(f1b <= 0.60).mean())

res = {"v11": evaluar(X11, modelos["A. Variables de la v11"]), "v12": evaluar(X12, modelos["B. A + variables por usuario"])}
tabla = pd.DataFrame({k: {"AUC-ROC": r["auc"], "AUC-PR": r["aucpr"], "F1": r["met"]["f1"],
                          "IC 95 % del F1": f"{r['ic'][0]:.3f} a {r['ic'][1]:.3f}", "p (F1 ≤ 0.60)": r["p"],
                          "Precisión": r["met"]["precision"], "Recall": r["met"]["recall"], "FPR (%)": 100 * r["met"]["fpr"],
                          "θ1": r["cal"]["theta1"], "θ2": r["cal"]["theta2"]} for k, r in res.items()})
tabla.columns = ["Versión 11", "Versión 12"]; tabla

,Versión 11,Versión 12
AUC-ROC,0.859,0.886
AUC-PR,0.634,0.688
F1,0.628,0.667
IC 95 % del F1,0.581 a 0.678,0.620 a 0.712
p (F1 ≤ 0.60),0.115,0.004
Precisión,0.851,0.879
Recall,0.498,0.537
FPR (%),0.711,0.607
θ1,0.21,0.22
θ2,0.56,0.95


**Tabla 3.** La versión 12 mejora todos los indicadores frente a la versión 11. El límite inferior del intervalo de confianza del F1 (0.620) supera la meta de 0.60, con p = 0.004. Además, el FPR disminuye de 0.71 % a 0.61 %, de modo que la mejora no se obtiene a costa de más falsas alarmas.

### 3.3 Distribución de las decisiones

In [5]:
nombres = {0: "Acceso directo", 1: "Verificación OTP", 2: "Bloqueo"}
dec, yt = res["v12"]["dec"], y[te]
pd.DataFrame({"Legítimos": [((dec == k) & (yt == 0)).sum() for k in nombres],
              "Sospechosos": [((dec == k) & (yt == 1)).sum() for k in nombres]}, index=nombres.values())

,Legítimos,Sospechosos
Acceso directo,19158,730
Verificación OTP,117,847
Bloqueo,0,0


In [6]:
p_t, c = res["v12"]["R"], res["v12"]["cal"]
v11_directo = np.where(p_t > c["theta2"], 2, np.where(p_t > c["theta1"], 1, 0))
comp = pd.DataFrame({"Política": ["Bloqueo directo sobre θ2 (v11)", "Escalonada con OTP priorizado (v12)"],
    "Sospechosos con OTP": [((v11_directo == 1) & (yt == 1)).sum(), ((dec == 1) & (yt == 1)).sum()],
    "Sospechosos bloqueados": [((v11_directo == 2) & (yt == 1)).sum(), ((dec == 2) & (yt == 1)).sum()],
    "Legítimos bloqueados": [((v11_directo == 2) & (yt == 0)).sum(), ((dec == 2) & (yt == 0)).sum()],
    "% sospechosos detenidos": [100 * ((v11_directo >= 1) & (yt == 1)).sum() / yt.sum(), 100 * ((dec >= 1) & (yt == 1)).sum() / yt.sum()]})
comp.round(1)

,Política,Sospechosos con OTP,Sospechosos bloqueados,Legítimos bloqueados,% sospechosos detenidos
0,Bloqueo directo sobre θ2 (v11),156,691,0,53.7
1,Escalonada con OTP priorizado (v12),847,0,0,53.7


**Tablas 4 y 5.** Con el mismo modelo, la política escalonada envía a OTP los accesos que antes se bloqueaban directamente. El porcentaje de sospechosos detenidos se mantiene en 53.7 % y no se produce ningún bloqueo directo: la detención se realiza mediante verificación y no mediante la interrupción del acceso.

### 3.4 Estabilidad frente a la semilla

In [7]:
filas = []
from run_validacion_pi13 import SEMILLAS
for s in SEMILLAS:
    m = ModeloAdaptAuth(seed=s).fit(X12[tr].to_numpy(), y[tr]); r = evaluar(X12, m)
    filas.append(dict(Semilla=s, F1=r["met"]["f1"], Precisión=r["met"]["precision"], Recall=r["met"]["recall"],
                      **{"FPR (%)": 100 * r["met"]["fpr"], "AUC-PR": r["aucpr"], "Bloqueos": int((r["dec"] == 2).sum())}))
sem = pd.DataFrame(filas)
pd.concat([sem, sem.drop(columns="Semilla").agg(["mean", "std"]).assign(Semilla=["Media", "DE"])]).round(3)

,Semilla,F1,Precisión,Recall,FPR (%),AUC-PR,Bloqueos
0,42,0.667,0.879,0.537,0.607,0.688,0.0
1,1,0.663,0.889,0.528,0.540,0.684,0.0
2,2,0.663,0.851,0.543,0.778,0.692,0.0
3,3,0.661,0.852,0.540,0.768,0.690,0.0
4,4,0.677,0.900,0.543,0.493,0.696,0.0
mean,Media,0.666,0.874,0.538,0.637,0.690,0.0
std,DE,0.007,0.022,0.006,0.131,0.004,0.0


**Tabla 6.** El F1 se mantiene entre 0.661 y 0.677 con las cinco semillas (DE = 0.007), sin bloqueos directos en ninguna corrida.

### 3.5 Comparación con la línea base y contribución de las variables

In [8]:
p = REPO / "results_validacion_pi13" / "validacion_pi13_sl.json"
v = json.loads(p.read_text()); comp_r, boot = v["comparacion_igual_fpr"], v["prueba_principal_bootstrap_usuario"]
reg, ada = comp_r["reglas_calibradas_igual_fpr"], comp_r["adaptauth_v12"]
fmt = lambda b, k=1: f"{k*b['media']:+.3f} ({k*b['ic95'][0]:.3f} a {k*b['ic95'][1]:.3f})"
display(pd.DataFrame({"Reglas (igual FPR)": [reg["recall"], 100 * reg["fpr"], reg["f1"]],
                      "AdaptAuth-ML v12": [ada["recall"], 100 * ada["fpr"], ada["f1"]],
                      "Diferencia (IC 95 %)": [fmt(boot["recall"]), fmt(boot["fpr"], 100), fmt(boot["f1"])]},
                     index=["Recall", "FPR (%)", "F1"]).round(3))
mc = v["mcnemar_sospechosos"]
print(f"McNemar exacto: {mc['solo_adaptauth']} sospechosos detectados solo por AdaptAuth-ML frente a {mc['solo_reglas']} solo por las reglas (p = {mc['p_exacto']:.1e}).")
imp = json.loads((REPO / "results_sl" / "metricas_sl.json").read_text())
pd.Series(imp["importancia_por_grupo"], name="Pérdida de AUC-PR al permutar").sort_values(ascending=False).round(4).to_frame()

,Reglas (igual FPR),AdaptAuth-ML v12,Diferencia (IC 95 %)
Recall,0.021,0.537,+0.517 (0.461 a 0.574)
FPR (%),1.287,0.607,-0.673 (-0.942 a -0.385)
F1,0.036,0.667,+0.631 (0.585 a 0.678)


McNemar exacto: 830 sospechosos detectados solo por AdaptAuth-ML frente a 16 solo por las reglas (p = 1.2e-221).


,Pérdida de AUC-PR al permutar
red_geo,0.558
comportamiento_usuario,0.084
autenticacion,0.059
temporal,0.007
historial,-0.005


**Tablas 7 y 8.** Frente a la línea base de reglas calibrada al mismo FPR de validación, la versión 12 incrementa el recall en 0.517 y reduce el FPR. Ambas diferencias son significativas por bootstrap de usuarios y por McNemar. En la importancia por permutación, el grupo de red y geolocalización sigue siendo el principal. El nuevo grupo de comportamiento por usuario ocupa el segundo lugar, por encima de autenticación y de las variables temporales.

## 4. Discusión

Las variables de comportamiento por usuario elevan el AUC-PR de validación de 0.589 a 0.682. Esa ganancia se traslada al periodo de prueba (0.634 a 0.688 de AUC-PR; 0.628 a 0.667 de F1). Por lo tanto, la mejora observada no es producto de ajustar el modelo al periodo de prueba, dado que la variante se eligió solo con validación y se evaluó una única vez. Este resultado concuerda con el enfoque de Freeman et al. (2016), que modela el riesgo como la rareza de cada atributo respecto del historial del propio usuario. También es coherente con el escenario CERT r4.2, cuyas variables comparan la actividad de cada empleado con su propio historial.

La política escalonada separa dos decisiones que la versión 11 unía: **detectar** un acceso sospechoso y **cómo detenerlo**. Al exigir OTP en lugar de bloquear, el usuario legítimo con riesgo alto puede continuar tras verificar su identidad, mientras que el atacante sin el segundo factor queda detenido igual que con un bloqueo. El bloqueo se reserva para la evidencia de ataque (OTP fallido o viaje imposible con riesgo alto).

## 5. Conclusiones

1. La versión 12 alcanza un F1 de 0.667 (IC 95 %: 0.620 a 0.712) en el periodo de prueba de SuspiciousLogin y supera la meta de 0.60 con significancia estadística (p = 0.004). Con ello, la hipótesis HE2 se sostiene.
2. Las ocho variables de comportamiento por usuario constituyen el segundo grupo de mayor importancia del modelo y mejoran la precisión, el recall y el FPR de manera simultánea.
3. La política escalonada mantiene la detención del 53.7 % de los accesos sospechosos y reduce a cero los bloqueos directos, priorizando la verificación OTP.

## Referencias

Freeman, D., Jain, S., Dürmuth, M., Biggio, B., & Giacinto, G. (2016). Who are you? A statistical approach to measuring user authenticity. En *Proceedings of the Network and Distributed System Security Symposium (NDSS 2016)*. Internet Society. https://doi.org/10.14722/ndss.2016.23240